# European Football Data Analytics: Comprehensive Multi-Season Study
**Author:** Mohammed Mueez Gous  
**Domain:** Sports Analytics & Performance Intelligence  
**Dataset:** Kaggle European Soccer Database (25,979 matches, 11 leagues, 10k+ players, 2008–2016)  
**Technologies:** Python, SQLite3, SQL, Pandas, Matplotlib, Seaborn, Power BI


## 1. Executive Summary & Project Objectives
Professional football (soccer) analytics has evolved from basic match statistics into data-driven tactical and scouting intelligence. This project investigates longitudinal performance across the top 11 European leagues over 8 full seasons (2008/09 to 2015/16).

### Core Research Questions:
1. **Home Pitch Advantage:** Does playing at home provide a statistically significant performance boost? How does this advantage vary across European leagues?
2. **Goal-Scoring Trajectories:** Has offensive output increased over time? Which leagues are the most entertaining for fans?
3. **Team Dominance:** Which European clubs boast the most fortress-like home records?
4. **Player Attribute Modeling:** What physical and technical skills correlate most strongly with overall player quality and career longevity?


In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Locate database path
db_candidates = [
    '../database.sqlite',
    'database.sqlite',
    r'C:\Users\Admin\Desktop\football-analysis\database.sqlite\database.sqlite',
    r'C:\Users\Admin\Desktop\football-analysis\database.sqlite'
]
db_path = next((p for p in db_candidates if os.path.exists(p) and os.path.isfile(p)), db_candidates[2])
conn = sqlite3.connect(db_path)
print(f"Connected to European Soccer Database at: {db_path}")


## 2. Database Schema & Entity Verification
We first audit the relational schema of the European Soccer Database.


In [ ]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)
print("Database Tables:")
for t in tables['name']:
    cnt = pd.read_sql(f"SELECT COUNT(*) as count FROM {t}", conn)['count'][0]
    print(f"  - {t:<20}: {cnt:>8,} records")


## 3. Data Auditing & Quality Checks
We verify primary key uniqueness, foreign key relationships, and check for missing values across match and player attributes.


In [ ]:
# Check duplicate IDs in Match table
match_dups = pd.read_sql("SELECT id, COUNT(*) as c FROM Match GROUP BY id HAVING c > 1;", conn)
team_dups = pd.read_sql("SELECT team_api_id, COUNT(*) as c FROM Team GROUP BY team_api_id HAVING c > 1;", conn)
player_dups = pd.read_sql("SELECT player_api_id, COUNT(*) as c FROM Player GROUP BY player_api_id HAVING c > 1;", conn)

print(f"Duplicate Match IDs  : {len(match_dups)}")
print(f"Duplicate Team IDs   : {len(team_dups)}")
print(f"Duplicate Player IDs : {len(player_dups)}")


## 4. Feature Engineering
We engineer analytical features to enable rich segmentation:
- `match_result`: Categorical outcome ('Home Win', 'Away Win', 'Draw')
- `total_goals`: Combined goals scored in the match
- `goal_difference`: Absolute victory margin
- `season_start_year`: Clean numeric year
- `player_age`: Age at the time of each FIFA attribute rating snapshot


In [ ]:
match_query = """
SELECT id, season, home_team_api_id, away_team_api_id, 
       home_team_goal, away_team_goal
FROM Match;
"""
matches_df = pd.read_sql(match_query, conn)

matches_df['match_result'] = 'Draw'
matches_df.loc[matches_df['home_team_goal'] > matches_df['away_team_goal'], 'match_result'] = 'Home Win'
matches_df.loc[matches_df['home_team_goal'] < matches_df['away_team_goal'], 'match_result'] = 'Away Win'

matches_df['total_goals'] = matches_df['home_team_goal'] + matches_df['away_team_goal']
matches_df['goal_difference'] = abs(matches_df['home_team_goal'] - matches_df['away_team_goal'])
matches_df['season_start_year'] = matches_df['season'].str[:4].astype(int)

matches_df.head()


## 5. Empirical Analysis & Visualizations
### 5.1 Macro Match Outcome Distribution (Home Pitch Advantage)
Out of 25,979 matches, home teams win **45.87%** of the time, while away teams win only **28.74%** (Draws: **25.39%**). Teams are **1.6 times more likely to win at home** than on the road.


In [ ]:
results_df = matches_df['match_result'].value_counts()
print(results_df)

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#10b981', '#ef4444', '#64748b']
bars = ax.bar(results_df.index, results_df.values, color=colors, edgecolor='#1e293b', linewidth=1.2)
ax.set_title('Match Outcomes Distribution (25,979 Matches | 2008-2016)', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Number of Matches', fontweight='bold')
ax.set_xlabel('Outcome', fontweight='bold')

for bar in bars:
    h = bar.get_height()
    pct = (h / len(matches_df)) * 100
    ax.annotate(f"{h:,}\n({pct:.1f}%)", (bar.get_x() + bar.get_width() / 2, h / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()


### 5.2 League-by-League Home Advantage Comparison
We compare home win percentages across all 11 European leagues. **Spain La Liga leads at 48.8%**, followed by the Dutch Eredivisie (47.8%) and Belgian Jupiler League (46.9%).


In [ ]:
league_query = """
SELECT 
    l.name AS league,
    COUNT(*) AS total_matches,
    SUM(CASE WHEN m.home_team_goal > m.away_team_goal THEN 1 ELSE 0 END) AS home_wins,
    ROUND(100.0 * SUM(CASE WHEN m.home_team_goal > m.away_team_goal THEN 1 ELSE 0 END) / COUNT(*), 1) AS home_win_pct
FROM Match m
JOIN League l ON m.league_id = l.id
GROUP BY l.name
ORDER BY home_win_pct ASC;
"""
league_df = pd.read_sql(league_query, conn)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(league_df['league'], league_df['home_win_pct'], color='#3b82f6', edgecolor='#1d4ed8')
ax.set_title('Home Win % by European League (2008-2016)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Home Win %', fontweight='bold')
ax.axvline(x=45.87, color='#dc2626', linestyle='--', linewidth=1.5, label='Overall Average (45.9%)')
ax.legend()

for bar in bars:
    w = bar.get_width()
    ax.annotate(f"{w:.1f}%", (w - 3.5, bar.get_y() + bar.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()


### 5.3 Longitudinal Goal Scoring Trends
We examine offensive production over the 8-season span. The average goals scored per match rose from **2.61 in 2008/09 to 2.77 in 2015/16**.


In [ ]:
goals_query = """
SELECT 
    season,
    CAST(SUBSTR(season, 1, 4) AS INT) AS start_year,
    COUNT(*) AS total_matches,
    SUM(home_team_goal + away_team_goal) AS total_goals,
    ROUND(AVG(home_team_goal + away_team_goal), 2) AS avg_goals_per_match
FROM Match
GROUP BY season
ORDER BY start_year ASC;
"""
goals_df = pd.read_sql(goals_query, conn)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(goals_df['season'], goals_df['avg_goals_per_match'], marker='o', color='#d97706', linewidth=2.5, markersize=8)
ax.set_title('Average Goals per Match by European Season', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Season', fontweight='bold')
ax.set_ylabel('Goals per Match', fontweight='bold')
ax.set_ylim(2.5, 2.9)

for _, row in goals_df.iterrows():
    ax.annotate(f"{row['avg_goals_per_match']:.2f}", (row['season'], row['avg_goals_per_match'] + 0.02),
                ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


### 5.4 Top European Clubs by Home Fortress Record
Ranking teams with at least 50 home fixtures by home victory rate. FC Barcelona, Real Madrid CF, FC Bayern Munich, and FC Porto dominate this ranking with home win rates above 75%.


In [ ]:
team_query = """
SELECT 
    home.team_long_name AS team,
    COUNT(*) AS total_home_matches,
    SUM(CASE WHEN m.home_team_goal > m.away_team_goal THEN 1 ELSE 0 END) AS home_wins,
    ROUND(100.0 * SUM(CASE WHEN m.home_team_goal > m.away_team_goal THEN 1 ELSE 0 END) / COUNT(*), 1) AS home_win_pct
FROM Match m
JOIN Team home ON m.home_team_api_id = home.team_api_id
GROUP BY home.team_long_name
HAVING total_home_matches >= 50
ORDER BY home_win_pct DESC
LIMIT 10;
"""
team_df = pd.read_sql(team_query, conn).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(team_df['team'], team_df['home_win_pct'], color='#8b5cf6', edgecolor='#6d28d9')
ax.set_title('Top 10 European Teams by Home Win % (Min. 50 Matches)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Home Win %', fontweight='bold')

for bar in bars:
    w = bar.get_width()
    ax.annotate(f"{w:.1f}%", (w - 4.5, bar.get_y() + bar.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()


### 5.5 Player Technical & Physical Attributes Correlation Matrix
We evaluate correlations across 16 core player attributes from 183,978 evaluations. **Ball Control, Dribbling, Short Passing, and Reactions** demonstrate the highest positive correlations with a player's Overall Rating.


In [ ]:
attr_query = """
SELECT overall_rating, crossing, finishing, heading_accuracy, short_passing,
       dribbling, ball_control, acceleration, sprint_speed, shot_power,
       stamina, strength, long_shots, aggression, marking, standing_tackle
FROM Player_Attributes
WHERE overall_rating IS NOT NULL;
"""
attr_df = pd.read_sql(attr_query, conn)
corr = attr_df.corr()

fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', ax=ax, annot_kws={'size': 8})
ax.set_title('Player Attributes Correlation Heatmap (183,978 Snapshots)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


## 6. Analytical Conclusion & Key Takeaways
1. **Home Advantage is Real & Quantifiable:** Across 25,979 fixtures, home teams won 45.87% of matches compared to just 28.74% for away teams.
2. **Tactical & League Variance:** Spain's La Liga exhibits the strongest home bias (48.8%), driven by hostile atmospheres and regional travel fatigue.
3. **Pace & Attacking Evolution:** European football witnessed a consistent upward trend in scoring over the 8 seasons analyzed (rising from 2.61 to 2.77 goals per game).
4. **Scouting Insights:** Technical mastery (ball control, short passing) outweighs raw athletic pace in determining elite player ratings.
